In [1]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone
from tqdm import tqdm

# --- 1. 定义路径 ---

# 输入目录 (包含 "Reach" 样本)
REACH_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach")

# 你为下一步指定的新输出目录
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\01_collect_max_time_range_for_every_coin\data")

# 输出的 JSON 统计文件名
OUTPUT_FILE = OUTPUT_DIR / "time_and_asset_summary.json"

# --- 2. 确保输出目录存在 ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出目录已准备好: {OUTPUT_DIR}")

# --- 3. 检查输入目录 ---
if not REACH_DIR.exists():
    print(f"错误：找不到输入目录 {REACH_DIR}")
    raise FileNotFoundError("Reach sample directory not found")

json_files = [f for f in os.listdir(REACH_DIR) if f.endswith('.json')]
if not json_files:
    print(f"警告：在 {REACH_DIR} 中未找到任何 .json 文件。")
    raise FileNotFoundError("No JSON files found in Reach directory")

print(f"--- 正在分析 {len(json_files)} 个 'Reach' 样本 ---")

# --- 4. 初始化统计变量 ---
min_last_action_timestamp = float('inf')
max_liquidation_timestamp = float('-inf')
all_asset_symbols = set()
files_processed = 0
files_failed = 0

# --- 5. 遍历所有 JSON 文件 ---
for filename in tqdm(json_files, desc="分析样本范围"):
    file_path = REACH_DIR / filename
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # 5a. 提取和比较时间戳
        # 将时间戳转换为浮点数以进行比较
        current_last_action_ts = float(data['last_action_timestamp'])
        current_liquidation_ts = float(data['liquidation_timestamp'])
        
        min_last_action_timestamp = min(min_last_action_timestamp, current_last_action_ts)
        max_liquidation_timestamp = max(max_liquidation_timestamp, current_liquidation_ts)
        
        # 5b. 提取资产符号
        snapshot = data.get('pre_liquidation_snapshot', [])
        if not snapshot:
            print(f"警告: {filename} 中 'pre_liquidation_snapshot' 为空。")
            
        for asset in snapshot:
            try:
                all_asset_symbols.add(asset['reserve']['symbol'])
            except KeyError:
                print(f"警告: {filename} 中某个资产缺少 'reserve' 或 'symbol' 键。")
        
        files_processed += 1
        
    except Exception as e:
        print(f"警告: 处理文件 {filename} 时出错: {e}")
        files_failed += 1

# --- 6. 整理并保存结果 ---

if files_processed > 0:
    # 转换为排序后的列表，以便 JSON 输出一致
    sorted_symbols = sorted(list(all_asset_symbols))
    
    # 转换为人类可读的日期时间 (UTC)
    earliest_dt = datetime.fromtimestamp(min_last_action_timestamp, timezone.utc).isoformat()
    latest_dt = datetime.fromtimestamp(max_liquidation_timestamp, timezone.utc).isoformat()

    # 创建汇总字典
    summary_data = {
        "total_samples_analyzed": files_processed,
        "samples_failed_to_process": files_failed,
        "time_range_for_price_query": {
            "earliest_timestamp_unix": min_last_action_timestamp,
            "latest_timestamp_unix": max_liquidation_timestamp,
            "earliest_datetime_utc": earliest_dt,
            "latest_datetime_utc": latest_dt,
            "note": "建议在查询外部 API 时在此范围前后增加一些缓冲（例如：前后各推几个月）。"
        },
        "unique_asset_symbols_found": sorted_symbols
    }
    
    print("\n--- 统计汇总 ---")
    print(f"最早的最后一次操作时间 (UTC): {earliest_dt} ({min_last_action_timestamp})")
    print(f"最晚的清算时间 (UTC):       {latest_dt} ({max_liquidation_timestamp})")
    print(f"涉及的唯一资产 ({len(sorted_symbols)} 种):")
    print(", ".join(sorted_symbols))

    # 写入 JSON 文件
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, indent=4)
        
    print(f"\n✅ 汇总统计信息已成功保存到: {OUTPUT_FILE}")

else:
    print("错误：没有成功处理任何文件，无法生成统计信息。")

输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\01_collect_max_time_range_for_every_coin\data
--- 正在分析 16 个 'Reach' 样本 ---


分析样本范围: 100%|██████████| 16/16 [00:00<00:00, 3924.27it/s]


--- 统计汇总 ---
最早的最后一次操作时间 (UTC): 2024-12-24T07:00:59+00:00 (1735023659.0)
最晚的清算时间 (UTC):       2025-10-30T13:37:13+00:00 (1761831433.0)
涉及的唯一资产 (10 种):
AAVE, EURC, GHO, USDC, USDbC, WETH, cbBTC, cbETH, weETH, wstETH

✅ 汇总统计信息已成功保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\01_collect_max_time_range_for_every_coin\data\time_and_asset_summary.json


## 试图获取区间的数据（测试）

In [4]:
import requests
import json
import pandas as pd

# --- 1. 设置你的 API 密钥 (来自你的示例) ---
API_KEY = 'CG-xtrHV9nWxQxtwq8tyThUtjNC' 

# --- 2. 准备 API 请求 (复用你的 headers) ---
headers = {
    'accept': 'application/json',
    'x-cg-demo-api-key': API_KEY
}

# --- 3. (已修正) 定义我们要测试的 10 个 ID ---
# 这是我们根据你的 coingecko_supported_coins.csv 文件找到的精确映射
# 我们将提取所有 ID 用于测试
COIN_ID_MAP = {
    "AAVE": "aave",
    "EURC": "euro-coin",
    "GHO": "gho",
    "USDC": "usd-coin",
    "USDbC": "bridged-usd-coin-base", # (已修正)
    "WETH": "weth",
    "cbBTC": "coinbase-wrapped-btc",    # (已修正)
    "cbETH": "coinbase-wrapped-staked-eth",
    "weETH": "wrapped-eeth",
    "wstETH": "wrapped-steth"          # (已修正)
}

# 将所有 ID 提取到一个列表中
ids_to_test_list = list(COIN_ID_MAP.values())

# CoinGecko API 接受用逗号分隔的 ID 字符串
ids_to_test_string = ",".join(ids_to_test_list)

# --- 4. 准备测试端点和参数 ---
# 我们使用 /simple/price 端点来测试 ID 是否有效
url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    'ids': ids_to_test_string,
    'vs_currencies': 'usd' # 我们想获取美元价格
}

print(f"--- 正在向 CoinGecko API 测试 {len(ids_to_test_list)} 个精确 ID ---")
print(f"测试的 ID: {ids_to_test_string}")

# --- 5. 执行测试请求 ---
try:
    # 发送 GET 请求 (注意：这次我们有 params)
    response = requests.get(url, headers=headers, params=params)
    
    # 检查请求是否成功
    response.raise_for_status() 
    
    print("\n请求成功！API 返回了数据。")
    data = response.json()
    
    print("\n--- API 原始返回 (JSON) ---")
    print(json.dumps(data, indent=2))
    
    # --- 6. 逐个核对结果 ---
    print("\n--- 逐个 ID 核对 ---")
    
    all_successful = True
    for symbol, coin_id in COIN_ID_MAP.items():
        # 检查返回的 JSON 中是否包含这个 ID，以及 'usd' 价格
        if coin_id in data and 'usd' in data[coin_id]:
            price = data[coin_id]['usd']
            # 使用 f-string 格式化对齐
            print(f"  ✅ {symbol:<6} (ID: {coin_id:<28}) - 成功! (当前价格: ${price})")
        else:
            print(f"  ❌ {symbol:<6} (ID: {coin_id:<28}) - 失败! API 响应中未找到。")
            all_successful = False
            
    if all_successful:
        print("\n--- 结论 ---")
        print("✅ 太好了！所有 10 个 ID 均已成功验证。")
        print("你现在可以安全地使用这个映射去执行上一个脚本（获取完整历史数据）了。")
    else:
        print("\n--- 结论 ---")
        print("⚠️ 警告：部分 ID 未能返回价格，请检查上面列表中的 '❌' 标记。")


# --- 7. 错误处理 (复用你的示例) ---
except requests.exceptions.HTTPError as http_err:
    if response.status_code == 401:
        print(f"HTTP 错误 401: API 密钥无效或未提供。请检查 'API_KEY' 变量。")
    elif response.status_code == 429:
        print(f"HTTP 错误 429: 超出速率限制。请稍后再试。")
    else:
        print(f"HTTP 错误: {http_err}")
except requests.exceptions.RequestException as req_err:
    print(f"请求失败: {req_err}")
except Exception as e:
    print(f"处理数据时发生错误: {e}")

--- 正在向 CoinGecko API 测试 10 个精确 ID ---
测试的 ID: aave,euro-coin,gho,usd-coin,bridged-usd-coin-base,weth,coinbase-wrapped-btc,coinbase-wrapped-staked-eth,wrapped-eeth,wrapped-steth

请求成功！API 返回了数据。

--- API 原始返回 (JSON) ---
{
  "aave": {
    "usd": 219.18
  },
  "bridged-usd-coin-base": {
    "usd": 0.99929
  },
  "coinbase-wrapped-btc": {
    "usd": 109899
  },
  "coinbase-wrapped-staked-eth": {
    "usd": 4239.83
  },
  "euro-coin": {
    "usd": 1.16
  },
  "gho": {
    "usd": 0.999297
  },
  "usd-coin": {
    "usd": 0.9998
  },
  "weth": {
    "usd": 3856.89
  },
  "wrapped-eeth": {
    "usd": 4164.99
  },
  "wrapped-steth": {
    "usd": 4695.26
  }
}

--- 逐个 ID 核对 ---
  ✅ AAVE   (ID: aave                        ) - 成功! (当前价格: $219.18)
  ✅ EURC   (ID: euro-coin                   ) - 成功! (当前价格: $1.16)
  ✅ GHO    (ID: gho                         ) - 成功! (当前价格: $0.999297)
  ✅ USDC   (ID: usd-coin                    ) - 成功! (当前价格: $0.9998)
  ✅ USDbC  (ID: bridged-usd-coin-base       ) - 成功!

## 每个币+ETH的区间波动

In [6]:
import requests
import json
import pandas as pd
from pathlib import Path
import time
from datetime import datetime, timezone
from tqdm import tqdm
import os
import math

# --- 1. 配置 ---

# 使用你已验证的 API 密钥
API_KEY = 'CG-xtrHV9nWxQxtwq8tyThUtjNC' 
HEADERS = {
    'accept': 'application/json',
    'x-cg-demo-api-key': API_KEY
}
CG_BASE_URL = "https://api.coingecko.com/api/v3"

# (已修正) 根据你的 CSV 找到的精确 CoinGecko ID 映射
COIN_ID_MAP = {
    "AAVE": "aave",
    "EURC": "euro-coin",
    "GHO": "gho",
    "USDC": "usd-coin",
    "USDbC": "bridged-usd-coin-base", # (已修正)
    "WETH": "weth",
    "cbBTC": "coinbase-wrapped-btc",    # (已修正)
    "cbETH": "coinbase-wrapped-staked-eth",
    "weETH": "wrapped-eeth",
    "wstETH": "wrapped-steth"          # (已修正)
}
# 我们也需要 ETH
COIN_ID_MAP["ETH"] = "ethereum"


# --- 2. 定义路径 ---

# 输入目录 (包含 "Reach" 样本)
REACH_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach")

# 你指定的新输出目录
OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\01_collect_max_time_range_for_every_coin\data\every_icon_price_sequence")

# 确保输出目录存在
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出目录已准备好: {OUTPUT_DIR}")

# --- 3. (新逻辑) 确定全局 90 天窗口 ---

print(f"\n--- 正在从 {REACH_DIR} 确定全局最晚清算时间... ---")

json_files = [f for f in os.listdir(REACH_DIR) if f.endswith('.json')]
if not json_files:
    raise FileNotFoundError(f"在 {REACH_DIR} 中未找到 'Reach' 样本。")

# 1. 找到所有样本中最晚的清算时间
global_max_end_ts = 0.0
for filename in json_files:
    file_path = REACH_DIR / filename
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        global_max_end_ts = max(global_max_end_ts, float(data['liquidation_timestamp']))
    except Exception as e:
        print(f"警告：处理文件 {filename} 时出错: {e}")

if global_max_end_ts == 0:
    raise ValueError("未能从样本中确定最晚清算时间。")

# 2. 以此时间为终点，计算 90 天前的起点
SEC_IN_90_DAYS = 90 * 86400
global_end_ts = int(global_max_end_ts)
global_start_ts = global_end_ts - SEC_IN_90_DAYS

# --- 4. 打印选区逻辑 ---
print("\n--- 选区逻辑 ---")
end_dt = datetime.fromtimestamp(global_end_ts, timezone.utc).isoformat()
start_dt = datetime.fromtimestamp(global_start_ts, timezone.utc).isoformat()
print(f"  -> 找到的最晚清算时间 (全局终点): {end_dt} ({global_end_ts})")
print(f"  -> 计算得到的 90 天前 (全局起点): {start_dt} ({global_start_ts})")
print(f"  -> 逻辑: 时间窗口为 90 天。CoinGecko API 将为所有币种返回 *小时* 粒度数据。")


# --- 5. 循环查询 CoinGecko (所有币种使用同一窗口) ---

print(f"\n--- 开始从 CoinGecko API 获取 {len(COIN_ID_MAP)} 个币种的 90 天小时数据 ---")

for symbol, coin_id in tqdm(COIN_ID_MAP.items(), desc="查询 CoinGecko"):
    
    # 准备 API 请求
    url = f"{CG_BASE_URL}/coins/{coin_id}/market_chart/range"
    params = {
        'vs_currency': 'usd',
        'from': global_start_ts,
        'to': global_end_ts
    }
    
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status() # 检查 HTTP 错误
        
        data = response.json()
        prices = data['prices']
        
        if not prices:
            print(f"  -> 警告: {symbol} 查询成功，但未返回价格数据。")
            continue
            
        # 处理返回的数据
        coin_data = []
        for ts_ms, price in prices:
            dt_utc = datetime.fromtimestamp(ts_ms / 1000, timezone.utc)
            coin_data.append({'datetime_utc': dt_utc.isoformat(), 'price_usd': price})
        
        # 保存单个币种的 CSV
        # 命名中加入 'hourly' 以示清晰
        df_coin = pd.DataFrame(coin_data)
        df_coin.to_csv(OUTPUT_DIR / f"{symbol}_hourly_price_data.csv", index=False, encoding='utf-8')
        
        tqdm.write(f"  -> 成功: {symbol:<6} - 获取 {len(prices)} 个数据点并保存到 {symbol}_hourly_price_data.csv")
        
    except requests.exceptions.HTTPError as e:
        tqdm.write(f"  -> 错误: {symbol:<6} 查询失败。状态码: {e.response.status_code}, 错误: {e.response.text}")
    except Exception as e:
        tqdm.write(f"  -> 错误: {symbol:<6} 处理失败: {e}")
    
    # 礼貌性暂停，避免触发速率限制
    time.sleep(1.5)

print(f"\n--- 全部完成 --- \n所有小时价格序列文件已保存到: {OUTPUT_DIR}")

输出目录已准备好: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\01_collect_max_time_range_for_every_coin\data\every_icon_price_sequence

--- 正在从 F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach 确定全局最晚清算时间... ---

--- 选区逻辑 ---
  -> 找到的最晚清算时间 (全局终点): 2025-10-30T13:37:13+00:00 (1761831433)
  -> 计算得到的 90 天前 (全局起点): 2025-08-01T13:37:13+00:00 (1754055433)
  -> 逻辑: 时间窗口为 90 天。CoinGecko API 将为所有币种返回 *小时* 粒度数据。

--- 开始从 CoinGecko API 获取 11 个币种的 90 天小时数据 ---


查询 CoinGecko:   0%|          | 0/11 [00:02<?, ?it/s]  

  -> 成功: AAVE   - 获取 2161 个数据点并保存到 AAVE_hourly_price_data.csv


查询 CoinGecko:   9%|▉         | 1/11 [00:06<00:40,  4.06s/it]  

  -> 成功: EURC   - 获取 2160 个数据点并保存到 EURC_hourly_price_data.csv


...

  -> 成功: wstETH - 获取 2161 个数据点并保存到 wstETH_hourly_price_data.csv


查询 CoinGecko:  91%|█████████ | 10/11 [00:37<00:03,  3.67s/it]  

  -> 成功: ETH    - 获取 2161 个数据点并保存到 ETH_hourly_price_data.csv


查询 CoinGecko: 100%|██████████| 11/11 [00:38<00:00,  3.54s/it]


--- 全部完成 --- 
所有小时价格序列文件已保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\01_collect_max_time_range_for_every_coin\data\every_icon_price_sequence
